# Explore the dataset

A quick look at what the public dataset contains: the recording metadata, one recording, and the experimenter
session notes. The last section scans all notes for identifiers (e-mail addresses, phone numbers, calendar dates)
so you can check what is shared.

Point `DATASET` at either
* the folder where you extracted the Dryad archives, or
* the folder with the downloaded Dryad files themselves (the ZIPs are read directly, nothing is extracted).

Requires `pandas` and `matplotlib` (`python -m pip install pandas matplotlib notebook`).

In [ ]:
import io
import os
import re
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

# Edit this, or set the CRP_DATASET environment variable before starting Jupyter.
DATASET = Path(os.environ.get("CRP_DATASET", "DATASET"))

DATA_FOLDERS = ("recordings", "currents", "annotations", "timing_quarantine", "native_currents")


def archive_for(rel):
    """Name of the Dryad ZIP that holds a dataset-relative path."""
    top, *rest = rel.split("/")
    if top in DATA_FOLDERS:
        return f"{top}_{rest[0]}.zip"
    if top in ("session_notes", "session_notes_clean"):
        return "session_notes.zip"
    return {"records.jsonl": "records.zip"}.get(rel, "code.zip" if top == "code" else None)


def open_file(rel):
    """Open a dataset file, from the extracted folder if present, otherwise from its ZIP.

    The ZIP is closed again after reading, so the notebook never keeps the downloaded files locked.
    """
    if (DATASET / rel).exists():
        return (DATASET / rel).open("rb")
    name = archive_for(rel)
    if name and (DATASET / name).exists():
        with zipfile.ZipFile(DATASET / name) as z:
            return io.BytesIO(z.read(rel))
    raise FileNotFoundError(f"{rel} not found in {DATASET}")


def read_text(rel):
    with open_file(rel) as f:
        return f.read().decode("utf-8")


def read_csv(rel, **kwargs):
    with open_file(rel) as f:
        return pd.read_csv(f, compression="gzip" if rel.endswith(".gz") else None, **kwargs)


print("Dataset folder:", DATASET.resolve())
print(sorted(p.name for p in DATASET.iterdir()))

## 1. Recording metadata

`metadata.csv` has one row per recording. Participants appear only as pseudonyms (`TFxx`); visits are given as a
visit number and days since the participant's first visit (`elapsed_visit_days`), never as calendar dates.

In [ ]:
meta = read_csv("metadata.csv", low_memory=False)
print(meta.shape[0], "recordings,", meta["participant_id"].nunique(), "participants,",
      meta["session_id"].nunique(), "sessions,", meta["visit_id"].nunique(), "visits")
meta.groupby("family").agg(recordings=("record_id", "size"), participants=("participant_id", "nunique"),
                           sessions=("session_id", "nunique"),
                           devices=("device_code", lambda s: ", ".join(sorted(s.unique()))))

In [ ]:
# Identity- and time-related columns, one row per visit.
cols = ["participant_id", "visit_id", "visit_number", "elapsed_visit_days", "family", "device_code", "ankle_code"]
meta[cols].drop_duplicates(["participant_id", "visit_id"]).sort_values(["participant_id", "visit_number"]).head(15)

In [ ]:
# Any value that looks like a calendar date (YYYY-MM-DD, YYYYMMDD, m/d/yyyy)?
date_like = re.compile(r"\b(?:19|20)\d{2}-[01]\d-[0-3]\d\b|\b(?:19|20)\d{2}[01]\d[0-3]\d\b|\b\d{1,2}/\d{1,2}/(?:19|20)\d{2}\b")
hits = {c: int(meta[c].astype(str).str.contains(date_like).sum()) for c in meta.columns}
print({c: n for c, n in hits.items() if n} or "no date-like values in metadata.csv")

## 2. One recording

Sensor, controller-parameter and state columns at 100 Hz. Units are those of each device's software (see README).

In [ ]:
FAMILY = "HK"   # VU, OSL, HK or Fox
row = meta[(meta["family"] == FAMILY) & (meta["status"] == "accepted")].iloc[0]
rec = read_csv(row["csv_path"])
print(row["csv_path"], "|", row["participant_id"], row["session_id"], row["device_code"], "|", len(rec), "rows")

window = rec[rec["time_s"] < 15]
fig, axes = plt.subplots(3, 1, figsize=(10, 6), sharex=True)
for ax, col in zip(axes, ["KneeAngle", "Fz", "state_labels"]):
    ax.plot(window["time_s"], window[col], lw=0.8)
    ax.set_ylabel(col)
axes[-1].set_xlabel("time (s)")
plt.tight_layout()

## 3. Session notes

Experimenter notes transcribed from the session forms. `session_notes/` holds the full text, `session_notes_clean/`
the same note without printed form text and unreadable OCR lines. `index.csv` links every note to sessions and
visits in `metadata.csv`.

In [ ]:
notes = read_csv("session_notes/index.csv")
print(len(notes), "notes covering", notes["session_ids"].str.split(";").explode().nunique(), "sessions")
display(notes.groupby("family").agg(notes=("note_id", "size"), participants=("participant_id", "nunique"),
                                    pages=("pages", "sum")))
notes.head()

In [ ]:
NOTE_ID = "N0005"   # change to open another note
note = notes.set_index("note_id").loc[NOTE_ID]
print(note.to_string(), "\n")
print(read_text(f"session_notes_clean/{NOTE_ID}.txt"))

In [ ]:
# The full transcription of the same note (first 60 lines), including the printed form.
print("\n".join(read_text(f"session_notes/{NOTE_ID}.txt").splitlines()[:60]))

In [ ]:
# Recordings from the sessions this note describes.
sessions = note["session_ids"].split(";")
meta[meta["session_id"].isin(sessions)].groupby(["session_id", "configuration_code", "status"]).size()

In [ ]:
def search_notes(pattern, folder="session_notes_clean", context=60):
    """Every match of a regular expression in the notes, with some surrounding text."""
    rows = []
    for nid, pid in zip(notes["note_id"], notes["participant_id"]):
        text = read_text(f"{folder}/{nid}.txt")
        for m in re.finditer(pattern, text, flags=re.I):
            snippet = re.sub(r"\s+", " ", text[max(0, m.start() - context):m.end() + context])
            rows.append({"note_id": nid, "participant_id": pid, "match": m.group(0), "context": snippet})
    return pd.DataFrame(rows)


hits = search_notes(r"socket")
print(len(hits), "matches in", hits["note_id"].nunique(), "notes")
hits.head(10)

In [ ]:
# Note days use the same scale as metadata.csv elapsed_visit_days: check same-day links agree.
same_day = notes[notes["link"].str.startswith("same_day")].copy()
same_day["visit_id"] = same_day["visit_ids"].str.split(";").str[0]
visit_days = meta.drop_duplicates("visit_id").set_index("visit_id")["elapsed_visit_days"]
same_day["visit_day"] = same_day["visit_id"].map(visit_days)
print((same_day["note_day"] == same_day["visit_day"]).mean().round(3), "of same-day notes match their visit day")

## 4. What identifying information do the notes contain?

The notes were de-identified before release: staff names became `[Staff member]`, e-mail addresses and phone
numbers were deleted, calendar dates became `[day N]` (days since the participant's first visit), dates printed on
blank forms became `[example date]`, unreadable dates were removed, and printed lab e-mails were dropped. The
cells below count those placeholders and scan every note for anything that still looks like an identifier.
Body measures and participant comments recorded in the sessions are kept on purpose.

In [ ]:
placeholders = r"\[(?:Staff member|day -?\d+|example date|date removed|month removed|removed|illegible|printed lab e-mail removed)\]"
counts = search_notes(placeholders, folder="session_notes", context=0)["match"].str.replace(r"\d+", "N", regex=True)
counts.value_counts()

In [ ]:
checks = {
    "e-mail address": r"[\w.+-]+@[\w-]+\.[a-z]{2,}",
    "phone number": r"(?<!\d)(?:\(\d{3}\)\s*|\d{3}[-.\s])\d{3}[-.]\d{4}(?!\d)",
    "date YYYYMMDD / YYYY-MM-DD": r"(?<!\d)(?:19|20)\d{2}-?(?:0[1-9]|1[0-2])-?(?:0[1-9]|[12]\d|3[01])(?!\d)",
    "date m/d/yyyy or m/d/yy": r"(?<![\d/.])\d{1,2}/\d{1,2}/(?:19|20)?\d{2}(?![\d/])",
    "month name + year": r"\b(?:jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec)[a-z]*\.?\s+(?:19|20)\d{2}\b",
}
for label, rx in checks.items():
    found = search_notes(rx, folder="session_notes", context=30)
    print(f"{label}: {len(found)}")
    if len(found):
        display(found.head())

Anything listed above deserves a look: the scans are deliberately broad, so they can also catch numbers that are
not dates (for example ratios such as `3/4/20` written as parameter values).